In [1]:
import json, os
from urllib.request import urlopen

In [2]:
# publications from your favourite IRMP author
author = 'Einstein, Albert'
url = 'https://inspirehep.net/api/literature?size=1000&q=a%20Albert.Einstein.1'
data = json.load(urlopen(url))

In [9]:
selected = []

for d in data['hits']['hits']:
    title = d['metadata']['titles'][0]['title']
    abstract = d['metadata']['abstracts'][0]['value']
    
    # journal
    pub = ''
    if 'publication_info' in d['metadata'] and len(d['metadata']['publication_info'])>0:
        pub = d['metadata']['publication_info'][0]
        
    
    # arxiv
    arxiv = ''
    if 'arxiv_eprints' in d['metadata'] and len(d['metadata']['arxiv_eprints'])>0:
        arxiv = d['metadata']['arxiv_eprints'][0]['value']
    
    # doi
    doi = ''
    if 'dois' in d['metadata'] and len(d['metadata']['dois'])>0:
        doi = d['metadata']['dois'][0]['value']
        
    # relevant publications, satisfying extra criterion
    if doi not in ['10.1007/3-7643-7436-5_7']:    
        selected.append({'pub':pub,'arxiv':arxiv,'doi':doi, 'title':title, 'abstract':abstract})

In [493]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver import ActionChains
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin

browser = webdriver.Chrome(service=webdriver.chrome.service.Service('/usr/bin/chromedriver'))

browser.get('https://research.dial.uclouvain.be/home')
wait = WebDriverWait(browser, timeout=2)

# login manually !!

In [12]:
i = -1

# iterate manually from here:

In [13]:
i += 1
i

0

In [14]:
paper = selected[i]
paper['doi']

'10.1126/science.84.2188.506'

In [ ]:
# Notes:
#- everything may not be clickable instantly, extra waiting time may be needed between cells
#- loading authors and various informations after identifiers have been inputed can take sometimes take a little while
#- scrolling to the right place seems crucial for many buttons/actions to be clickable

In [1084]:
# add

## click add

sec = browser.find_element(By.CSS_SELECTOR,'div[id="header-navbar-wrapper"]')

buts = sec.find_elements(By.CSS_SELECTOR,'button')

for b in buts:
    if( b.text=='Add a submission'):
        b.click()
        browser.implicitly_wait(.3)
        break

## wait
wait.until(lambda _ : len(browser.find_elements(By.CSS_SELECTOR,'button[title="Publication"]'))>0 )

        
## select UCLouvain Publication

sec = browser.find_element(By.CLASS_NAME,'modal-dialog')

but = sec.find_element(By.CSS_SELECTOR,'button[title="Publication"]')

but.click()
browser.implicitly_wait(.5)

In [1085]:
# publication section


## wait
wait.until(lambda _ : len(browser.find_elements(By.ID,'section_publication-type'))>0 )
ActionChains(browser).scroll_by_amount(0, 200).perform()

pubdiv = browser.find_element(By.ID,'section_publication-type')

## publication type

typediv = pubdiv.find_element(By.CSS_SELECTOR,'div[aria-label="Publication type"]')
ActionChains(browser).scroll_to_element(typediv).perform()
typediv.click()
browser.implicitly_wait(5)

article = pubdiv.find_element(By.CSS_SELECTOR,'button[title="Journal article"]')
article.click()
browser.implicitly_wait(1000)

## wait
wait.until(lambda _ : any([s.is_displayed() for s in pubdiv.find_elements(By.CSS_SELECTOR,'div[id="label_dc_type_subtype"]')]))


## publication subtype

subtypedivs = pubdiv.find_elements(By.CSS_SELECTOR,'div[aria-label="Document subtype"]')

for tmp in subtypedivs:
    if tmp.is_displayed():
        ActionChains(browser).scroll_to_element(tmp).perform()
        tmp.click()
        browser.implicitly_wait(500)
        break

wait.until(lambda _ : any([s.is_displayed() for s in pubdiv.find_elements(By.CSS_SELECTOR,'button[title="None"]')]))

buts = pubdiv.find_elements(By.CSS_SELECTOR,'button[title="None"]')

for tmp in buts:
    if tmp.is_displayed():
        ActionChains(browser).scroll_to_element(tmp).perform()
        tmp.click()
        browser.implicitly_wait(.5)
        break

In [1086]:
ActionChains(browser).scroll_by_amount(0, 200).perform()

In [1087]:
# identifiers

sec = browser.find_element(By.ID,'section_publication-identifiers')

## doi
doibox = sec.find_element(By.CSS_SELECTOR,'input[name="dc.identifier.doi"]')
doibox.clear()
doibox.send_keys(paper['doi'])
browser.implicitly_wait(.3)

## arxiv

### select arxiv ID
idbox = sec.find_elements(By.CSS_SELECTOR,'select[name="dc.identifier_QUALDROP_METADATA"]')

for tmp in idbox:
    if tmp.is_displayed():
        select = Select(tmp)
        browser.implicitly_wait(.3)
        break

select.select_by_value('3: dc.identifier.arxiv')

### input arxiv ID
idbox = sec.find_elements(By.CSS_SELECTOR,'input[name="dc.identifier_QUALDROP_VALUE"]')
for tmp in idbox:
    if tmp.is_displayed():
        tmp.clear()
        tmp.send_keys(paper['arxiv'])
        browser.implicitly_wait(.3)
        break

# random click and click again to trigger update
head = browser.find_element(By.CSS_SELECTOR,'div[id="publication-identifiers-header"]')
head.click()
head.click()
browser.implicitly_wait(.3)


In [1088]:
ActionChains(browser).scroll_by_amount(0, 200).perform()

In [1089]:
# authors
sec = browser.find_element(By.CSS_SELECTOR,'div[id="section_publication-authors"]')


## wait
wait.until(lambda _ : len(sec.find_elements(By.CLASS_NAME,'author-name'))>0)

spans = sec.find_elements(By.CLASS_NAME,'author-name')

found = False
for s in spans:
    if s.text == author:
        ActionChains(browser).scroll_to_element(s).perform()
        ActionChains(browser).scroll_by_amount(0,100).perform()
        s.click()
        browser.implicitly_wait(.3)
        found = True
        break

    
## for a long author list, I may not appear, then add
if not found:
    but = sec.find_elements(By.CSS_SELECTOR,'button[title="Add more"]')

    for b in but:
        if(b.is_displayed()):
            ActionChains(browser).scroll_to_element(b).perform()
            ActionChains(browser).scroll_by_amount(0,100).perform()
            b.click()
            browser.implicitly_wait(1)
            break

## dialog box
sec = browser.find_element(By.CLASS_NAME,'modal-dialog')


## reset author, just in case
box = sec.find_element(By.CSS_SELECTOR,'input[id="dc_contributor_author"]')

box.clear()
box.send_keys(author)
browser.implicitly_wait(500)

## search author

# but = sec.find_element(By.CSS_SELECTOR,'i[title="Search"]')
# but.click()

wait.until(lambda _: any([l.text==author for l in sec.find_elements(By.CSS_SELECTOR,'label')]))

lab = sec.find_elements(By.CSS_SELECTOR,'label')

for l in lab:
    if l.text == author:
        l.click()
        browser.implicitly_wait(5)
        break

## close
sec = browser.find_element(By.CLASS_NAME,'modal-footer')

but = sec.find_elements(By.CSS_SELECTOR,'button')

for b in but:
    if b.text in ['Set', 'Add']:
        b.click()
        browser.implicitly_wait(.3)
        break

In [1090]:
ActionChains(browser).scroll_by_amount(0, 400).perform()

In [1091]:
# basic data

sec = browser.find_element(By.ID,'section_publication-base')

box = sec.find_element(By.CSS_SELECTOR,'input[name="dc.title"]')

box.clear()
box.send_keys(paper['title'])
browser.implicitly_wait(5)

## abstract
ActionChains(browser).scroll_by_amount(0, 400).perform()

box = sec.find_element(By.CSS_SELECTOR,'textarea[name="dc.description.abstract"]')

box.clear()
box.send_keys(paper['abstract'])
browser.implicitly_wait(500)

In [1092]:
ActionChains(browser).scroll_by_amount(0, 400).perform()

In [1093]:
## affiliation popup

sub = sec.find_element(By.ID,'affiliation-drag-list')

but = sub.find_element(By.CSS_SELECTOR,'button[title="Add more"]')

but.click()
browser.implicitly_wait(.3)

### institution

drop = browser.find_element(By.CSS_SELECTOR,'div[aria-label="Affiliation institution"]')

drop.click()
browser.implicitly_wait(.3)

but = browser.find_element(By.CSS_SELECTOR,'button[title="UCLouvain"]')

but.click()
browser.implicitly_wait(.3)

### institute

drop = browser.find_element(By.CSS_SELECTOR,'div[aria-label="Affiliation entity"]')

drop.click()
browser.implicitly_wait(.3)

but = browser.find_element(By.CSS_SELECTOR,'button[title="SST/IRMP - Institut de recherche en mathématique et physique"]')

but.click()
browser.implicitly_wait(.3)

### validate

subsec = browser.find_element(By.CLASS_NAME,'modal-dialog')

buts = subsec.find_elements(By.CSS_SELECTOR,'button')

if buts[-1].text == 'Add':
    buts[-1].click()
else:
    raise

In [1094]:
ActionChains(browser).scroll_by_amount(0, 600).perform()

In [1095]:
# specific data

sec = browser.find_element(By.ID,'section_publication-specific')
ActionChains(browser).scroll_to_element(sec).perform()
ActionChains(browser).scroll_by_amount(0, -200).perform()

## published

drop = sec.find_element(By.CSS_SELECTOR,'input[name="publication.publicationStatus"]')

drop.click()
browser.implicitly_wait(5)

but = sec.find_element(By.CSS_SELECTOR,'button[title="Published"]')

but.click()
browser.implicitly_wait(5)

## peer reviewed

but = sec.find_element(By.CSS_SELECTOR,'input[name="publication.serial.peerReviewed"]')
if not but.is_selected():
    but.click()
    browser.implicitly_wait(5)


## volume, issue, page, date
ActionChains(browser).scroll_by_amount(0, 200).perform()

box = sec.find_elements(By.CSS_SELECTOR,'input[name="publication.serial.volume"]')

for b in box:
    if b.is_displayed():
        b.clear()
        b.send_keys(paper['pub']['journal_volume'])
        browser.implicitly_wait(.3)
        break

ActionChains(browser).scroll_by_amount(0, 100).perform()
box = sec.find_elements(By.CSS_SELECTOR,'input[name="publication.serial.pages"]')

for b in box:
    if b.is_displayed():
        b.clear()
        if 'artid' in paper['pub']:
            pages = paper['pub']['artid']
        elif 'page_start' in paper['pub'] and 'page_end':
            pages = '{:}-{:}'.format(paper['pub']['page_start'],paper['pub']['page_end'])
        else:
            pages = 'n/a'
        b.send_keys(pages)
        browser.implicitly_wait(.3)
        break

ActionChains(browser).scroll_by_amount(0, 100).perform()
box = sec.find_elements(By.CSS_SELECTOR,'input[id="publication_serial_dateIssued"]')

for b in box:
    if b.is_displayed():
        b.clear()
        b.send_keys(paper['pub']['year'])
        browser.implicitly_wait(.3)
        break

In [1097]:
ActionChains(browser).scroll_by_amount(0, 200).perform()

In [1098]:
# funding

sec = browser.find_element(By.ID,'section_publication-funding')

but = sec.find_element(By.CSS_SELECTOR,'button[title="Add more"]')

but.click()
browser.implicitly_wait(.3)

### popup: just select FNRS and quit

sec = browser.find_element(By.CLASS_NAME,'modal-dialog')

drop = sec.find_element(By.CSS_SELECTOR,'div[aria-label="Organization name"]')

drop.click()
browser.implicitly_wait(.3)

but = sec.find_element(By.CSS_SELECTOR,'button[title="F.R.S.-FNRS - Fund for Scientific Research [BE]"]')

but.click()
browser.implicitly_wait(.3)

### quit

sub = sec.find_element(By.CLASS_NAME,'modal-footer')

buts = sub.find_elements(By.CSS_SELECTOR,'button')

for b in buts:
    if b.text=='Add':
        b.click()
        browser.implicitly_wait(3)
        break

In [1099]:
ActionChains(browser).scroll_by_amount(0, 400).perform()

In [1100]:
# license

but = browser.find_element(By.CSS_SELECTOR,'input[id="granted"]')
ActionChains(browser).scroll_to_element(but).perform()
browser.implicitly_wait(3)
if not but.is_selected():
    but.click()

In [1101]:
# upload file

## get arxiv pdf

url = 'https://arxiv.org/pdf/'+paper['arxiv']

req = urlopen(url)

out = req.read()

fname = '/home/albert/snap/chromium/3464/{:}.pdf'.format(paper['arxiv'])

with open(fname,'wb') as file:
    file.write(out)
    file.close()

## upload it

sec = browser.find_element(By.CSS_SELECTOR,'div[id="section_upload"]')

filedrop = sec.find_element(By.CSS_SELECTOR,'input[type="file"]')

filedrop.send_keys(fname)

## delete file

os.remove(fname)
browser.implicitly_wait(.3)

In [484]:
### if something failed, save for later instead of submitting

# sec = browser.find_element(By.CSS_SELECTOR,'div.submission-form-footer')

# but = sec.find_element(By.CSS_SELECTOR,'button[id="saveForLater"]')

# but.click()
# browser.implicitly_wait(.3)

In [1102]:
# submit

sec = browser.find_element(By.CSS_SELECTOR,'div.submission-form-footer')

but = sec.find_element(By.CSS_SELECTOR,'button[id="deposit"]')

but.click()
browser.implicitly_wait(.3)